# 蛋白配列 vs PDB構造の1次元アラインメント(汎用)

アラインメント自体は`chem.protein.sequence_align()`(UniProt canonical配列の取得、PDB構造からの
配列抽出とcanonicalへのアラインメントをまとめて行うライブラリ関数)に任せ、このノートブックは
その結果をどう表示するか(色付きHTML)に専念する、という設計にしてある。

対象の蛋白・PDB構造はStep 1の設定セルを書き換えるだけで差し替えられる。ここでは例として
`ADRB2_HUMAN`(β2アドレナリン受容体, `P07550`)を使う: インバースアゴニスト・アンタゴニスト・
フル/部分アゴニスト(内因性リガンドのアドレナリンを含む)・正/負のアロステリックモジュレーター・
細胞内アロステリック部位のアンタゴニスト・Gs蛋白複合体構造など、薬理学的に多様な18構造を並べる。

## Step 1: 設定

ここを書き換えれば別の蛋白・別のPDB構造セットに差し替えられる。値はそのまま
`chem.protein.sequence_align()`の引数として渡す。

- `UNIPROT_ACCESSION`: 対象蛋白のUniProtアクセッション
- `CANONICAL_FEATURE_TYPE` / `CANONICAL_FEATURE_DESCRIPTION`: canonical配列として切り出す
  feature(例: シグナルペプチドを除いた`Chain`、特定の`Domain`)。`CANONICAL_FEATURE_TYPE=None`なら
  full-lengthのUniProt配列をそのまま使う。
- `MARKER_FEATURE_TYPES`: マーカー表示するUniProt feature typeのタプル。酵素なら`"Active site"`、
  リーダードメインなら`"Site"`、受容体のリガンド結合ポケットなら`"Binding site"`など、対象に応じて
  変える。`None`(デフォルト)ならマーカーなし。
- `PDB_IDS`: 比較したいPDB構造のリスト(WT・変異体・電子密度が一部欠けている構造、など任意に混在可)

In [ ]:
# ---- 設定: ここを書き換えれば別の蛋白・別の構造セットで使える ----

UNIPROT_ACCESSION = "P07550"  # ADRB2_HUMAN, beta-2 adrenergic receptor

CANONICAL_FEATURE_TYPE = None  # シグナルペプチド切断なし、full-lengthをそのまま使う
CANONICAL_FEATURE_DESCRIPTION = None

MARKER_FEATURE_TYPES = ("Binding site",)  # オルソステリックポケットの結合部位残基

PDB_IDS = [
    "2RH1",  # inverse agonist: carazolol (T4L fusion, 最初の高分解能構造)
    "3NY8",  # inverse agonist: ICI 118,551
    "5JQH",  # inverse agonist: carazolol + 不活性型安定化ナノボディNb60
    "3NYA",  # neutral antagonist: alprenolol
    "3PDS",  # covalent (irreversible) agonist
    "3P0G",  # agonist-bound, active state stabilizing nanobody
    "4LDO",  # agonist: adrenaline (内因性リガンド) + ナノボディ
    "6MXT",  # agonist: salmeterol (長時間作動性)
    "3SN6",  # agonist + Gs蛋白複合体(活性型、ヌクレオチドフリー三者複合体)
    "7DHR",  # agonist: isoprenaline + Gs複合体 (cryo-EM)
    "6KR8",  # full agonist bound state
    "6OBA",  # negative allosteric modulator
    "7BZ2",  # agonist: formoterol + Gs複合体 (cryo-EM)
    "7DHI",  # partial agonist: salbutamol + Gs複合体 (cryo-EM)
    "6PS3",  # antagonist/beta-blocker: carvedilol (XFEL, ligand exchange)
    "6PS5",  # antagonist/beta-blocker: propranolol (XFEL, ligand exchange)
    "5X7D",  # inverse agonist carazolol + 細胞内アロステリック部位のアンタゴニスト
    "6N48",  # agonist BI167107 + 正のアロステリックモジュレーター
]
DATA_DIR = "sequence_alignment_data"

## Step 2: PDB構造のダウンロード

`PDB_IDS`で指定した構造を`chem.rcsb.download_structures`でまとめて取得する。
`chem.protein.sequence_align()`はローカルのPDB/CIFファイルパスを受け取る設計(ダウンロードは
`chem.rcsb`側の責務)なので、ここでファイルパスのリストも作っておく。

In [ ]:
import os

from chem.rcsb.fetch import download_structures

download_structures(PDB_IDS, outdir=DATA_DIR, filetype="pdb")
structure_paths = [os.path.join(DATA_DIR, f"{pdb_id}.pdb") for pdb_id in PDB_IDS]

## Step 3: `chem.protein.sequence_align()`でアラインメント情報を取得

UniProtからのcanonical配列取得、各PDB構造でcanonicalに最も良く合う鎖の自動選択(T4リゾチーム融合・
ナノボディ・G蛋白サブユニットなど、余分な鎖が混ざっていても正しい鎖を選ぶ)、そしてcanonical座標への
アラインメント(繰り返し配列付近での同点タイブレークも含む)を、この1関数呼び出しにまとめてある。
戻り値は辞書で、以降のセル(Step 4以降)はそこから取り出したデータを使って表示するだけの
「カスタムスクリプト」という位置づけ。

In [ ]:
import os

import chem.protein
import numpy as np
import pandas as pd

result = chem.protein.sequence_align(
    UNIPROT_ACCESSION,
    structure_paths,
    canonical_feature_type=CANONICAL_FEATURE_TYPE,
    canonical_feature_description=CANONICAL_FEATURE_DESCRIPTION,
    marker_feature_types=MARKER_FEATURE_TYPES,
)

canonical_seq = result["canonical_seq"]
marker_pos = result["marker_positions"]

# result["sequences"] is keyed by file path; re-key by PDB id for readability
# in the rest of the notebook.
observed = {os.path.splitext(os.path.basename(p))[0]: seq for p, seq in result["sequences"].items()}

print(f"{result['protein_name']} ({result['organism']})")
if CANONICAL_FEATURE_TYPE:
    print(
        f"{CANONICAL_FEATURE_TYPE} '{CANONICAL_FEATURE_DESCRIPTION}': "
        f"precursor residues {result['feature_start']}-{result['feature_end']} "
        f"({result['feature_end'] - result['feature_start'] + 1} aa)"
    )
print(f"Marker positions (canonical numbering, 1-based): {sorted(marker_pos)}")
print()
print("Canonical sequence:")
print(canonical_seq)


def pairwise_identity(seq_a, seq_b):
    """Fraction of positions with an identical residue, over positions both
    sequences actually have a residue at -- gaps don't count in either the
    numerator or the denominator, matching the identity convention used
    elsewhere in chem.protein (see structural_align.identity_matrix()).
    """
    matched = [(a, b) for a, b in zip(seq_a, seq_b) if a != "-" and b != "-"]
    return sum(a == b for a, b in matched) / len(matched) if matched else 0.0


# Pairwise identity across every protein, canonical (WT) included -- all are
# already on the same canonical-indexed coordinates, so no realignment is
# needed here, just a direct position-by-position comparison.
sequences_by_label = {"Canonical": canonical_seq, **observed}
labels = list(sequences_by_label)
identity_matrix = pd.DataFrame(
    [[pairwise_identity(sequences_by_label[a], sequences_by_label[b]) for b in labels] for a in labels],
    index=labels,
    columns=labels,
).round(3)

# The diagonal (self-identity, always 1.0) isn't informative -- blank it out
# to "-" so it doesn't visually compete with genuine 1.0 off-diagonal hits
# (identical sequences between two different structures), which get
# highlighted instead. to_numpy(copy=True) sidesteps pandas' copy-on-write
# read-only array views, which a plain .values/.copy() can still hit.
values = identity_matrix.to_numpy(dtype=float, copy=True)
np.fill_diagonal(values, np.nan)
display_matrix = pd.DataFrame(values, index=labels, columns=labels)

print()
print("Pairwise identity matrix (yellow = identical, excluding self):")
display_matrix.style.format("{:.3f}", na_rep="-").map(
    lambda v: "background-color: yellow" if v == 1.0 else ""
)

## Step 4: 色付きHTML表示

`IPython.display.HTML`で等幅フォントを保ったまま装飾する版。ブロック先頭に10残基おきの位置ルーラー
(番号+`|`)を添え、相違点は赤太字、canonicalにない挿入(ギャップ)は灰色イタリック、マーカー位置
(活性部位・結合部位など)は太字+下線、で区別する。

`reference`引数で基準をcanonical以外の構造に変えられる(例: `reference="2RH1"`)。位置は引き続き
canonical基準の座標を使うので、マーカー位置の意味は変わらない。基準にした構造自身が欠損している
位置(T4リゾチーム融合部分やN/C末端など)では比較対象がないため、赤字にはせずそのまま表示する。

In [ ]:
import html as html_lib

from IPython.display import HTML, display


def _ruler_lines(start, end, prefix):
    """A two-line position ruler for canonical columns [start, end) (0-based,
    end exclusive): a right-aligned number every 10 residues, and a "|" tick
    directly under it -- e.g.:
            10        20        30
             |         |         |
    """
    width = end - start
    numbers = [" "] * width
    ticks = [" "] * width
    for pos in range(start + 1, end + 1):  # 1-based canonical position
        if pos % 10 == 0:
            i = pos - start - 1
            ticks[i] = "|"
            s = str(pos)
            for j, ch in enumerate(s):
                idx = i - len(s) + 1 + j
                if 0 <= idx < width:
                    numbers[idx] = ch
    return [f'{prefix}{"".join(numbers)}', f'{prefix}{"".join(ticks)}']


def render_alignment_html(observed, canonical_seq, marker_pos, width=100, reference=None):
    if reference is None:
        ref_seq, ref_label, others = canonical_seq, "Canonical", observed
    else:
        ref_seq, ref_label, others = observed[reference], f"Reference ({reference})", {
            l: o for l, o in observed.items() if l != reference
        }

    label_width = max(len(l) for l in list(others) + [ref_label, "Marker"]) + 1
    blank_prefix = " " * (label_width + 2)

    def marker_span(esc):
        return f'<b><u>{esc}</u></b>'

    blocks = []
    for start in range(0, len(canonical_seq), width):
        end = min(start + width, len(canonical_seq))

        ref_html = "".join(
            marker_span(html_lib.escape(r)) if (i + 1) in marker_pos else html_lib.escape(r)
            for i, r in enumerate(ref_seq[start:end], start=start)
        )
        rows = _ruler_lines(start, end, blank_prefix) + [f'{ref_label:<{label_width}}: {ref_html}']

        for label, obs in others.items():
            cells = []
            for i, (r, o) in enumerate(zip(ref_seq[start:end], obs[start:end]), start=start):
                esc = html_lib.escape(o)
                is_marker = (i + 1) in marker_pos
                if o == "-":
                    cells.append(f'<span style="color:#999999; font-style:italic;">{esc}</span>')
                elif r == "-":
                    # reference has no residue here -- nothing to diff against.
                    cells.append(marker_span(esc) if is_marker else esc)
                elif o != r:
                    deco = " text-decoration:underline;" if is_marker else ""
                    cells.append(f'<span style="color:#e74c3c; font-weight:bold;{deco}">{esc}</span>')
                elif is_marker:
                    cells.append(marker_span(esc))
                else:
                    cells.append(esc)
            rows.append(f'{label:<{label_width}}: {"".join(cells)}')
        blocks.append("\n".join(rows))

    body = "\n\n".join(blocks)
    display(HTML(f'<pre style="font-family: ui-monospace, monospace; line-height:1.4;">{body}</pre>'))


render_alignment_html(observed, canonical_seq, marker_pos)

## Step 5: 2RH1(最初の高分解能構造)を基準にした表示

canonicalの代わりに`2RH1`自身を基準にして、他の構造との相違だけを見る。

In [ ]:
render_alignment_html(observed, canonical_seq, marker_pos, reference="2RH1")